# 09. Area predictions, calendar context and ELUM farmland

Optional review after 04 and 07. For each reference boundary it shows the predicted labels of the saved Paoay maps, compares them with the crop calendars written in the KMZ area notes, and adds the NAMRIA Existing Land Use 2023 farmland.

**Needs:** the Paoay 04 crop baseline, optionally the Paoay 07 aligned map for the current Training 02 model, the Paoay 06 land-activity map, the KMZ files, `atok_landuse_path` (ELUM 2023 GeoPackage) and a calendar CSV (`paoay_calendar_file`, default `review/calendars/kmz_notes_calendar.csv`).

Nothing is refitted, masked or relabelled. The calendars say when farmers report a crop in the field, not which crop occupied a polygon in 2024, so calendar agreement is context, not accuracy. ELUM farmland is a land-use designation and does not name crops.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

import matplotlib.pyplot as plt
import rasterio
from IPython.display import display

from crop_pipeline.analysis_workflow import resolve_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline.paoay import aligned, area_review

settings = json.loads((root / "settings.local.json").read_text())

def local(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (root / path).resolve()

paoay_root = local(settings.get("report_root", "./data/pipeline_work/inventory")).parent / "paoay"
calendar_file = local(settings.get("paoay_calendar_file", paoay_root / "review/calendars/kmz_notes_calendar.csv"))
landuse_path = local(settings["atok_landuse_path"])
kmz_root = local(settings.get("paoay_kmz_root", "./data/paoay"))
kmz = [kmz_root / name for name in ("00 - Region of Interest.kmz", "00 - Atok Boundary.kmz", "Crop Area.kmz")]
roi_name = settings.get("paoay_roi_name") or "Region of Interest"
context_names = [name for name in settings.get("paoay_context_names", []) if name != roi_name]

## 1. Saved maps

The fixed map covers January–April 2024; the aligned map uses each pixel's own four-month window from its green-up month. The aligned map is included only when Paoay 07 has run for the current Training 02 model. (The annual-model map is an experiment; see `experiments/notebooks/annual/`.)

In [ ]:
baseline_handoff = resolve_stage_pointer(paoay_root / "crop_baseline/crop_baseline_stage.json")
products = [area_review.product("fixed", baseline_handoff)]
aligned_folder = area_review.aligned_bundle_for(paoay_root / "aligned_exploratory", aligned.default_model(root))
if aligned_folder is not None:
    products.append(area_review.product("aligned", aligned_folder))
else:
    print("No Paoay 07 bundle for the current aligned model; run 07 to add it.")
grid = products[0]["grid"]
if any(item["grid"] != grid for item in products):
    raise ValueError("The saved Paoay maps use different grids")
roi, areas, _ = aligned.study_area(kmz, roi_name, context_names, grid["crs"])
masks = area_review.area_masks(areas, grid)
study = area_review.area_masks([{"name": "study", "geometry": roi}], grid)["study"]
output = paoay_root / "area_review" / baseline_handoff.parent.name
output.mkdir(parents=True, exist_ok=True)
show_table([{"map": item["name"], "period": item["period"], "source": Path(item["path"]).relative_to(root)}
            for item in products], title="Maps compared")

## 2. Predicted labels inside each boundary

One row per reference area and one column per map, each zoomed to its boundary. Colours inside the black outline are that map's labels; surrounding pixels are faded. Titles give the most frequent confident crop and the share of the area with a confident prediction (top probability ≥ 0.60 and margin ≥ 0.10).

In [ ]:
landuse = area_review.read_landuse(landuse_path, roi.buffer(200).bounds, grid["crs"])
farmland = area_review.farmland(landuse)
figure = area_review.plot_area_panels(products, areas, study, farmland)
figure.savefig(output / "area_panels.png", dpi=150)
display(figure)
plt.close(figure)
rows = [row for item in products for row in area_review.composition(item, masks)]
area_review.write_csv(output / "area_labels.csv", rows)
show_table([{key: row[key] for key in ("product", "area", "label", "area_ha", "percent_of_mapped")}
            for row in rows if row["pixels"]], title="Labels by area (hectares and % of mapped pixels)")

## 3. Calendar context compared with predictions

Entries come from the KMZ notes: the months each crop is reported in the field, from planting to harvest. A confident pixel is *calendar-consistent* when its crop has an entry for that area whose months overlap the months its prediction describes. Both maps' windows check timing as well as the crop name. Seedbeds, unmodelled crops (lettuce, radish, Chinese cabbage and others) and unconfirmed entries are listed but never count as consistent. All reference areas lie in Atok.

In [ ]:
entries = area_review.read_calendar(calendar_file, set(masks))
comparison = [row for item in products for row in area_review.calendar_comparison(item, masks, entries)]
detail = [row for item in products for row in area_review.calendar_detail(item, masks, entries)]
area_review.write_csv(output / "calendar_comparison.csv", comparison)
area_review.write_csv(output / "calendar_detail.csv", detail)
figure = area_review.plot_calendar(entries, comparison, products)
figure.savefig(output / "calendar_comparison.png", dpi=150)
display(figure)
plt.close(figure)
show_table([{key: row[key] for key in ("product", "area", "calendar_crops_in_period", "other_or_context_crops",
                                       "confident_pixels", "dominant_prediction", "dominant_percent_of_confident",
                                       "calendar_consistent_percent_of_confident", "status")} for row in comparison],
           title="Calendar context vs predictions (context only, not accuracy)")
print("Calendar file:", calendar_file.relative_to(root) if calendar_file.is_relative_to(root) else calendar_file.name)

## 4. NAMRIA ELUM 2023 farmland

Farmland is the ELUM sub-classes Crops, Orchard, Agroforestry and Strategic Agriculture and Fisheries Development Zone. General Production Agriculture is left out because the ELUM metadata uses it for closed forest on alienable land; pasture and grazing land are left out because they are not cropped. The first table shows which ELUM classes each reference polygon falls in. The label shares compare farmland with the Paoay 06 active-cropland class from the multi-year land-activity map.

In [ ]:
show_table(area_review.landuse_by_area(landuse, areas), title="ELUM 2023 classes inside each reference area")
farm_mask = area_review.polygon_mask(farmland, grid)
land_activity = resolve_stage_pointer(paoay_root / "land_activity/land_activity_stage.json").parent / "land_activity.tif"
with rasterio.open(land_activity) as source:
    if source.crs.to_string() != grid["crs"] or tuple(source.transform)[:6] != tuple(grid["transform"]):
        raise ValueError("The land-activity map is on a different grid from the crop maps")
    activity_cropland = source.read(1) == 1  # active cropland
farm_rows = [row for item in products
             for row in area_review.farmland_composition(item, study, farm_mask, activity_cropland, "Land-activity cropland")]
area_review.write_csv(output / "farmland_labels.csv", farm_rows)
show_table([{key: row[key] for key in ("product", "area", "label", "area_ha", "percent_of_mapped")}
            for row in farm_rows if row["pixels"]], title="Labels inside and outside ELUM farmland")
figure = area_review.plot_farmland(products, farmland, farm_rows, areas)
figure.savefig(output / "farmland_paoay.png", dpi=150)
display(figure)
plt.close(figure)
print("Outputs:", output.relative_to(root))

## Interpretation

These are preliminary maps without dated local labels. A crop named in an area's calendar can still be absent from that polygon in 2024, and a confident prediction can fall on a crop the model was never trained on (lettuce, radish, Chinese cabbage). ELUM farmland and the cropland layers bound where the three-crop model is meaningful; they do not validate its labels. The January 2027 field visit (Paoay 05) remains the independent check.